# PE-Core-L-14-336 — dựng chỉ mục keyframe đầy đủ trên Colab (1x T4)

**Chỉ chạy notebook này nếu cổng pre-test đã GO** — xem `docs/PRETEST_ENCODER.md`.

- **Model**: `timm/PE-Core-L-14-336` qua `open_clip` (embedding dim **1024**)
- **Tiền xử lý**: resize ép vuông (**squash**) 336×336 bilinear — đúng preprocess mặc định
  của model VÀ đúng giao thức pre-test (`scripts/pretest_pe_core.py` §3.3); **không** center-crop.
- **Thứ tự hàng**: khớp **từng hàng** với `data/metadata.json` (177.321 hàng) — cùng thứ tự
  với `embeddings_siglip2_384.npy`, nên mọi mã đọc chỉ mục cũ dùng lại được nguyên xi.
- **Đầu ra**: `embeddings_pe_core_l336.npy` (177321 × 1024, float32, ~726 MB) lưu vào Google Drive.
- **Resume**: tiến độ ghi vào memmap + file `.progress` trên Drive — đứt phiên chạy lại là tiếp tục.
- **Văn bản (khi tích hợp)**: text tower PE-Core có context **32 token** (ngắn hơn SigLIP 64);
  câu dài bị cắt — cân nhắc cắt khúc như đường SigLIP sản xuất. Đọc kết luận vi/en trong
  `docs/PRETEST_ENCODER.md` §5 trước khi chọn prompt.


In [ ]:
# 1. Kết nối Google Drive & cài thư viện
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi
!apt-get update -qq && apt-get install -y -qq aria2
!pip install -q -U open_clip_torch hf_transfer
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'


In [ ]:
# 2. Tải 15 file ZIP keyframes chính thức của BTC (tự thử lại khi lỗi mạng)
import os, time, subprocess

ZIP_DIR = "/content/zips"
os.makedirs(ZIP_DIR, exist_ok=True)

ZIP_URLS = [
    "https://aic-data.ledo.io.vn/map-keyframes-aic25-b1.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L21.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L22.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L23.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L24.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L25.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L26_a.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L26_b.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L26_c.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L26_d.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L26_e.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L27.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L28.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L29.zip",
    "https://aic-data.ledo.io.vn/Keyframes_L30.zip",
]

for url in ZIP_URLS:
    fname = os.path.basename(url)
    dest = os.path.join(ZIP_DIR, fname)
    if os.path.exists(dest) and os.path.getsize(dest) > 1024 * 1024:
        print(f"  bỏ qua (đã có): {fname}")
        continue
    for attempt in range(1, 4):
        rc = subprocess.call([
            "aria2c", "-x", "8", "-s", "8", "--retry-wait=5", "--max-tries=5",
            "-d", ZIP_DIR, "-o", fname, "--allow-overwrite=true", url])
        if rc == 0:
            break
        print(f"  ! {fname}: aria2 rc={rc}, thử lại {attempt}/3"); time.sleep(10)
    else:
        raise RuntimeError(f"tải hỏng: {url}")
print("Tải xong toàn bộ ZIP.")


In [ ]:
# 3. Giải nén toàn bộ keyframes
import glob, os

KEYFRAMES_DIR = "/content/keyframes"
os.makedirs(KEYFRAMES_DIR, exist_ok=True)

zips = sorted(glob.glob("/content/zips/Keyframes_*.zip"))
print(f"{len(zips)} file ZIP keyframes. Đang giải nén...")
for zp in zips:
    print(f"  -> {os.path.basename(zp)} ({os.path.getsize(zp)/1e6:.0f} MB)")
    !unzip -q -o "{zp}" -d {KEYFRAMES_DIR}/
print("Giải nén xong.")


In [ ]:
# 4. Tải metadata.json (thứ tự hàng CHUẨN của chỉ mục) & kiểm đủ 177.321 ảnh
import json, os, urllib.request

META_URL = ("https://huggingface.co/datasets/BaeBaeBoo1010/aic2026-keyframes"
            "/resolve/main/metadata.json")
META_PATH = "/content/metadata.json"
if not os.path.exists(META_PATH):
    urllib.request.urlretrieve(META_URL, META_PATH)
metadata = json.load(open(META_PATH))
print(f"metadata: {len(metadata):,} hàng")
assert len(metadata) == 177321, "metadata lệch số hàng!"

# định vị thư mục từng video trên đĩa (cấu trúc trong ZIP có thể lồng thêm 1 cấp)
KEYFRAMES_DIR = "/content/keyframes"
video_dir = {}
for root, dirs, _files in os.walk(KEYFRAMES_DIR):
    for d in dirs:
        if d.startswith("L") and "_V" in d:
            video_dir[d] = os.path.join(root, d)

paths, missing = [], []
for m in metadata:
    vd = video_dir.get(m["video_id"])
    p = os.path.join(vd, m["frame_filename"]) if vd else None
    if p and os.path.exists(p):
        paths.append(p)
    else:
        missing.append(m["rel_path"]); paths.append(None)
print(f"đủ ảnh: {len(paths) - len(missing):,} / {len(metadata):,}")
if missing:
    print("VÍ DỤ THIẾU:", missing[:10])
    raise RuntimeError(f"THIẾU {len(missing)} ảnh — đừng encode, chỉ mục sẽ lệch hàng!")


In [ ]:
# 5. Encode toàn bộ bằng PE-Core-L-14-336 (fp16 autocast, batch 64, resumable)
import os, time
import numpy as np
import torch
import open_clip
from PIL import Image
from torch.utils.data import Dataset, DataLoader

MODEL_NAME = "hf-hub:timm/PE-Core-L-14-336"
DIM = 1024
N = len(paths)
BATCH = 64
DRIVE_DIR = "/content/drive/MyDrive"
MM_PATH = os.path.join(DRIVE_DIR, "embeddings_pe_core_l336_tmp.dat")
PROG_PATH = os.path.join(DRIVE_DIR, "embeddings_pe_core_l336.progress")
OUT_DRIVE = os.path.join(DRIVE_DIR, "embeddings_pe_core_l336.npy")

device = "cuda" if torch.cuda.is_available() else "cpu"
model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME)
model = model.to(device).eval()
# Giao thức pre-test: preprocess mặc định của model = squash 336×336 bilinear.
# (kiểm bằng mắt — transform phải là Resize((336, 336)), KHÔNG CenterCrop)
print(preprocess)

class KF(Dataset):
    def __init__(self, paths, start):
        self.paths, self.start = paths, start
    def __len__(self):
        return len(self.paths) - self.start
    def __getitem__(self, i):
        p = self.paths[self.start + i]
        try:
            im = Image.open(p).convert("RGB")
        except Exception:
            im = Image.new("RGB", (336, 336))   # ảnh hỏng: vector của ảnh đen
        return preprocess(im)

start = 0
if os.path.exists(PROG_PATH):
    start = int(open(PROG_PATH).read().strip() or 0)
    print(f"RESUME từ hàng {start:,}")
mm = np.lib.format.open_memmap(
    MM_PATH, mode=("r+" if start else "w+"), dtype=np.float32, shape=(N, DIM))

dl = DataLoader(KF(paths, start), batch_size=BATCH, num_workers=2, pin_memory=True)
t0, done = time.time(), start
with torch.inference_mode():
    for xb in dl:
        xb = xb.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            v = model.encode_image(xb)
        v = v / v.norm(dim=-1, keepdim=True)
        mm[done:done + len(v)] = v.float().cpu().numpy()
        done += len(v)
        if done % (BATCH * 50) < BATCH:
            rate = (done - start) / max(time.time() - t0, 1e-9)
            eta = (N - done) / max(rate, 1e-9) / 60
            print(f"  {done:,}/{N:,}  ({rate:.0f} ảnh/s, còn ~{eta:.0f} phút)", flush=True)
            mm.flush(); open(PROG_PATH, "w").write(str(done))
mm.flush(); open(PROG_PATH, "w").write(str(done))
print(f"Encode XONG {done:,} hàng sau {(time.time()-t0)/60:.0f} phút.")

# đổi memmap tạm thành .npy chuẩn trên Drive
os.replace(MM_PATH, OUT_DRIVE)
print(f"Đã lưu {OUT_DRIVE} ({os.path.getsize(OUT_DRIVE)/1e6:.0f} MB)")


In [ ]:
# 6. Kiểm tra nhanh sau khi encode (trước khi tải về / trỏ hệ thống vào)
import numpy as np, os
OUT_DRIVE = "/content/drive/MyDrive/embeddings_pe_core_l336.npy"
E = np.load(OUT_DRIVE, mmap_mode="r")
print("shape:", E.shape, E.dtype)
assert E.shape == (177321, 1024)
norms = np.linalg.norm(np.asarray(E[::5000], dtype=np.float32), axis=1)
print("norm mẫu (phải ~1.0):", norms.min().round(4), norms.max().round(4))
zero = int((np.abs(np.asarray(E[::5000])).sum(axis=1) < 1e-6).sum())
print("hàng rỗng trong mẫu (phải 0):", zero)
print("Tải về máy: dùng nút Files bên trái Colab, hoặc files.download(OUT_DRIVE)")
